In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
df_raw = pd.read_csv("data/nba_playoffs_2024_cleaned.csv")
print(f'Raw dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'Columns: {df_raw.columns.tolist()}')


In [ ]:
stat_cols = ['MP','FG','FGA','FG%','3P','3PA','3P%','2P','2PA',
             'FT','FTA','FT%','ORB','DRB','TRB','AST','STL','BLK',
             'TOV','PF','PTS']
df = df_raw[df_raw['G'] >= 5].reset_index(drop=True)
print(f'After filtering (G >= 5): {len(df)} players retained')
print(f'Dropped: {len(df_raw) - len(df)} players')
df[stat_cols] = df[stat_cols].fillna(0)
print(f'Missing values after imputation: {df[stat_cols].isnull().sum().sum()}')

In [ ]:
clean_cols = ['Rk','Player','Pos','Age','Tm','G','GS'] + stat_cols
df[clean_cols].to_csv('data/nba_playoffs_2024_cleaned.csv', index=False)
print('Cleaned CSV saved: data/nba_playoffs_2024_cleaned.csv')

In [ ]:
df[stat_cols].describe().round(2)

In [ ]:
print('Position counts:')
print(df['Pos'].value_counts().to_string())

In [ ]:
X = df[stat_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'X_scaled shape : {X_scaled.shape}')
print(f'Mean (should be ~0): {X_scaled.mean(axis=0).round(4)[:5]}...')
print(f'Std  (should be ~1): {X_scaled.std(axis=0).round(4)[:5]}...')

In [ ]:
corr_matrix = np.corrcoef(X_scaled.T)
n, p = X_scaled.shape

chi2_stat = -(n - 1 - (2*p + 5)/6) * np.log(np.linalg.det(corr_matrix))
df_chi2   = p * (p - 1) // 2
p_val     = 1 - stats.chi2.cdf(chi2_stat, df_chi2)

print('Bartlett\'s Test of Sphericity')
print(f'  chi² = {chi2_stat:,.1f}')
print(f'  df   = {df_chi2}')
print(f'  p    = {p_val:.2e}')
print()
if p_val < 0.001:
    print('✓ Significant (p < 0.001): correlation matrix is NOT identity.')
    print('  → PCA and Factor Analysis are appropriate for this dataset.')

In [ ]:
key_stats = ['PTS','AST','TRB','STL','BLK','3PA','FGA','MP','TOV']
corr_sub = pd.DataFrame(X_scaled, columns=stat_cols)[key_stats].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#1A1D27')

im = ax.imshow(corr_sub.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(key_stats))); ax.set_xticklabels(key_stats, color='white', rotation=45, ha='right')
ax.set_yticks(range(len(key_stats))); ax.set_yticklabels(key_stats, color='white')
for i in range(len(key_stats)):
    for j in range(len(key_stats)):
        ax.text(j, i, f'{corr_sub.values[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white')
ax.set_title('Correlation Matrix (Key Stats)', color='white', fontsize=11, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

explained  = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)
eigenvalues = pca_full.explained_variance_

print('PC   Eigenvalue   Var %   Cumulative %')
print('-'*45)
for i, (ev, e, c) in enumerate(zip(eigenvalues[:10], explained[:10], cumulative[:10]), 1):
    marker = ' ← retain' if ev >= 1 else ''
    print(f'PC{i:2d}   {ev:8.3f}   {e*100:5.2f}%   {c*100:6.2f}%{marker}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('#0F1117')

for ax in axes:
    ax.set_facecolor('#1A1D27')
    ax.tick_params(colors='#CCCCCC')
    for sp in ax.spines.values(): sp.set_edgecolor('#333344')

n_show = 10
# Left: variance explained per component
ax = axes[0]
ax.bar(range(1, n_show+1), explained[:n_show]*100, color='#4CC9F0', edgecolor='#0F1117', zorder=3)
ax.plot(range(1, n_show+1), cumulative[:n_show]*100, 'o-', color='#F72585', lw=2, ms=5, label='Cumulative %')
ax.axhline(80, color='#FFD60A', ls='--', lw=1.2, label='80% threshold')
ax.set_xlabel('Principal Component', color='#AAAAAA')
ax.set_ylabel('Variance Explained (%)', color='#AAAAAA')
ax.set_title('Scree Plot', color='white', fontsize=11)
ax.set_xticks(range(1, n_show+1))
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=8)
ax.grid(axis='y', color='#333344', lw=0.5, zorder=0)

# Right: eigenvalues with Kaiser line
ax = axes[1]
ax.bar(range(1, n_show+1), eigenvalues[:n_show], color='#7B2FBE', edgecolor='#0F1117', zorder=3)
ax.axhline(1, color='#FFD60A', ls='--', lw=1.5, label='Kaiser criterion (λ=1)')
ax.set_xlabel('Principal Component', color='#AAAAAA')
ax.set_ylabel('Eigenvalue (λ)', color='#AAAAAA')
ax.set_title('Eigenvalues – Kaiser Criterion', color='white', fontsize=11)
ax.set_xticks(range(1, n_show+1))
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=8)
ax.grid(axis='y', color='#333344', lw=0.5, zorder=0)

plt.suptitle('PCA – Variance & Eigenvalue Analysis', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

n_kaiser = sum(eigenvalues >= 1)
print(f'Components with eigenvalue ≥ 1 (Kaiser criterion): {n_kaiser}')
print(f'Variance explained by {n_kaiser} PCs: {cumulative[n_kaiser-1]*100:.1f}%')

In [ ]:
pca3 = PCA(n_components=3)
scores = pca3.fit_transform(X_scaled)

loadings = pd.DataFrame(
    pca3.components_.T,
    index=stat_cols,
    columns=['PC1','PC2','PC3']
)

print('PCA Component Loadings (3 components):')
print(f'Variance explained: PC1={explained[0]*100:.1f}%, PC2={explained[1]*100:.1f}%, PC3={explained[2]*100:.1f}%')
print(f'Total: {cumulative[2]*100:.1f}%')
print()
loadings.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0F1117')

pc_titles = [
    f'PC1 ({explained[0]*100:.1f}%) — Overall Production',
    f'PC2 ({explained[1]*100:.1f}%) — Perimeter vs Interior',
    f'PC3 ({explained[2]*100:.1f}%) — Defensive Specialist'
]

for i, (ax, title) in enumerate(zip(axes, pc_titles)):
    ax.set_facecolor('#1A1D27')
    ax.tick_params(colors='#CCCCCC', labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#333344')

    col = f'PC{i+1}'
    sorted_idx = loadings[col].abs().sort_values(ascending=False).index
    vals = loadings.loc[sorted_idx, col]
    bar_colors = ['#4CC9F0' if v > 0 else '#F72585' for v in vals]

    ax.barh(range(len(sorted_idx)), vals, color=bar_colors, edgecolor='#0F1117', lw=0.4)
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels(sorted_idx, color='#CCCCCC')
    ax.axvline(0, color='white', lw=0.8)
    ax.axvline(0.3, color='#555566', lw=0.6, ls='--')
    ax.axvline(-0.3, color='#555566', lw=0.6, ls='--')
    ax.set_xlabel('Loading', color='#AAAAAA')
    ax.set_title(title, color='white', fontsize=9, pad=6)
    ax.grid(axis='x', color='#333344', lw=0.4)

plt.suptitle('PCA Component Loadings', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pos_colors = {'PG':'#E63946','SG':'#F4A261','SF':'#2A9D8F','PF':'#457B9D','C':'#6A0572'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0F1117')

for ax in axes:
    ax.set_facecolor('#1A1D27')
    ax.tick_params(colors='#CCCCCC')
    for sp in ax.spines.values(): sp.set_edgecolor('#333344')

# PC1 vs PC2
ax = axes[0]
for pos, col in pos_colors.items():
    mask = df['Pos'] == pos
    ax.scatter(scores[mask, 0], scores[mask, 1], c=col, label=pos,
               s=45, alpha=0.85, edgecolors='white', linewidths=0.3, zorder=3)

# Annotate top scorers
highlights = ['LeBron James','Nikola Jokic','Luka Doncic','Stephen Curry','Giannis Antetokounmpo']
for name in highlights:
    idx_list = df[df['Player']==name].index.tolist()
    if idx_list:
        idx = idx_list[0]
        ax.annotate(name.split()[-1], (scores[idx,0], scores[idx,1]),
                    fontsize=7.5, color='#FFD60A',
                    xytext=(5,3), textcoords='offset points',
                    fontweight='bold')

ax.axhline(0, color='#333344', lw=0.8); ax.axvline(0, color='#333344', lw=0.8)
ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%) – Overall Production', color='#AAAAAA')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%) – Perimeter vs Interior', color='#AAAAAA')
ax.set_title('PC1 vs PC2 by Position', color='white', fontsize=11)
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=9, ncol=2)
ax.grid(color='#333344', lw=0.4, zorder=0)

# PC1 vs PC3
ax = axes[1]
for pos, col in pos_colors.items():
    mask = df['Pos'] == pos
    ax.scatter(scores[mask, 0], scores[mask, 2], c=col, label=pos,
               s=45, alpha=0.85, edgecolors='white', linewidths=0.3, zorder=3)
ax.axhline(0, color='#333344', lw=0.8); ax.axvline(0, color='#333344', lw=0.8)
ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%) – Overall Production', color='#AAAAAA')
ax.set_ylabel(f'PC3 ({explained[2]*100:.1f}%) – Defensive Specialist', color='#AAAAAA')
ax.set_title('PC1 vs PC3 by Position', color='white', fontsize=11)
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=9, ncol=2)
ax.grid(color='#333344', lw=0.4, zorder=0)

plt.suptitle('PCA Score Plots – 2023-24 NBA Playoffs', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_scores = df[['Player','Pos','Tm','PTS','AST','TRB']].copy()
df_scores['PC1'] = scores[:, 0]
df_scores['PC2'] = scores[:, 1]
df_scores['PC3'] = scores[:, 2]

print('Top 10 players by PC1 (Overall Production):')
df_scores.sort_values('PC1', ascending=False).head(10)[['Player','Pos','PTS','AST','TRB','PC1']].round(2)

In [ ]:
fa = FactorAnalysis(n_components=3, rotation='varimax', random_state=42)
fa.fit(X_scaled)
fa_scores   = fa.transform(X_scaled)
fa_loadings = pd.DataFrame(
    fa.components_.T,
    index=stat_cols,
    columns=['F1','F2','F3']
)

# Variance statistics
ss        = (fa_loadings**2).sum(axis=0)
prop_var  = ss / len(stat_cols)
cumul_var = prop_var.cumsum()
communalities = (fa_loadings**2).sum(axis=1)

var_df = pd.DataFrame({
    'SS Loadings' : ss.round(3),
    'Prop. Var'   : prop_var.round(3),
    'Cumul. Var'  : cumul_var.round(3)
})
print('Factor Variance Summary:')
print(var_df.to_string())
print(f'\nTotal variance explained by 3 factors: {cumul_var["F3"]*100:.1f}%')

In [ ]:
print('Factor Loadings (|value| ≥ 0.50 highlighted):')

def highlight_loading(val):
    if abs(val) >= 0.70:
        return 'background-color: #1a472a; color: white; font-weight: bold'
    elif abs(val) >= 0.50:
        return 'background-color: #2d6a4f; color: white'
    return ''

fa_loadings.round(3).style.applymap(highlight_loading)

In [ ]:
comm_df = pd.DataFrame({'Communality': communalities.round(3)}).sort_values('Communality', ascending=False)
print('Communalities (proportion of each variable\'s variance explained by 3 factors):')
comm_df

In [ ]:
factor_titles = [
    'F1 (37.2%) — Scoring & Paint Presence',
    'F2 (17.4%) — Perimeter Playmaking',
    'F3 (16.2%) — Rebounding & Interior Defense'
]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.patch.set_facecolor('#0F1117')

for i, (ax, title) in enumerate(zip(axes, factor_titles)):
    ax.set_facecolor('#1A1D27')
    ax.tick_params(colors='#CCCCCC', labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#333344')

    col = f'F{i+1}'
    sorted_idx = fa_loadings[col].abs().sort_values(ascending=False).index
    vals = fa_loadings.loc[sorted_idx, col]

    # Color by strength
    bar_colors = []
    for v in vals:
        if abs(v) >= 0.70: bar_colors.append('#00B4D8' if v > 0 else '#E63946')
        elif abs(v) >= 0.50: bar_colors.append('#4CC9F0' if v > 0 else '#F72585')
        else: bar_colors.append('#778DA9')

    ax.barh(range(len(sorted_idx)), vals, color=bar_colors, edgecolor='#0F1117', lw=0.4)
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels(sorted_idx, color='#CCCCCC')
    ax.axvline(0, color='white', lw=0.8)
    ax.axvline(0.5, color='#FFD60A', lw=0.8, ls='--', alpha=0.6)
    ax.axvline(-0.5, color='#FFD60A', lw=0.8, ls='--', alpha=0.6)
    ax.set_xlabel('Factor Loading', color='#AAAAAA')
    ax.set_title(title, color='white', fontsize=9.5, pad=6)
    ax.grid(axis='x', color='#333344', lw=0.4)

strong_patch = mpatches.Patch(color='#00B4D8', label='|loading| ≥ 0.70')
mod_patch    = mpatches.Patch(color='#4CC9F0', label='|loading| ≥ 0.50')
weak_patch   = mpatches.Patch(color='#778DA9', label='|loading| < 0.50')
fig.legend(handles=[strong_patch, mod_patch, weak_patch],
           loc='lower center', ncol=3, fontsize=9,
           facecolor='#1A1D27', labelcolor='white', edgecolor='#333344',
           bbox_to_anchor=(0.5, -0.06))

plt.suptitle('Factor Analysis Loadings – Varimax Rotation', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0F1117')

for ax in axes:
    ax.set_facecolor('#1A1D27')
    ax.tick_params(colors='#CCCCCC')
    for sp in ax.spines.values(): sp.set_edgecolor('#333344')

ax = axes[0]
for pos, col in pos_colors.items():
    mask = df['Pos'] == pos
    ax.scatter(fa_scores[mask, 0], fa_scores[mask, 1], c=col, label=pos,
               s=45, alpha=0.85, edgecolors='white', linewidths=0.3)
ax.axhline(0, color='#333344', lw=0.8); ax.axvline(0, color='#333344', lw=0.8)
ax.set_xlabel('F1 – Scoring & Paint Presence', color='#AAAAAA')
ax.set_ylabel('F2 – Perimeter Playmaking', color='#AAAAAA')
ax.set_title('FA: F1 vs F2 by Position', color='white', fontsize=11)
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=9, ncol=2)
ax.grid(color='#333344', lw=0.4)

ax = axes[1]
for pos, col in pos_colors.items():
    mask = df['Pos'] == pos
    ax.scatter(fa_scores[mask, 0], fa_scores[mask, 2], c=col, label=pos,
               s=45, alpha=0.85, edgecolors='white', linewidths=0.3)
ax.axhline(0, color='#333344', lw=0.8); ax.axvline(0, color='#333344', lw=0.8)
ax.set_xlabel('F1 – Scoring & Paint Presence', color='#AAAAAA')
ax.set_ylabel('F3 – Rebounding & Interior Defense', color='#AAAAAA')
ax.set_title('FA: F1 vs F3 by Position', color='white', fontsize=11)
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=9, ncol=2)
ax.grid(color='#333344', lw=0.4)

plt.suptitle('Factor Analysis Score Plots – 2023-24 NBA Playoffs', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.patch.set_facecolor('#0F1117')

for ax in axes:
    ax.set_facecolor('#1A1D27')
    ax.tick_params(colors='#CCCCCC', labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('#333344')

# PC1
ax = axes[0]
pc1_sorted = loadings['PC1'].sort_values(ascending=True)
ax.barh(range(len(pc1_sorted)), pc1_sorted,
        color=['#4CC9F0' if v>0 else '#F72585' for v in pc1_sorted])
ax.set_yticks(range(len(pc1_sorted))); ax.set_yticklabels(pc1_sorted.index, color='#CCCCCC')
ax.axvline(0, color='white', lw=0.8)
ax.set_title(f'PC1 ({explained[0]*100:.1f}%) — Overall Production\n[PCA — No Rotation]',
             color='white', fontsize=10)
ax.set_xlabel('Loading', color='#AAAAAA')
ax.grid(axis='x', color='#333344', lw=0.4)

# F1
ax = axes[1]
f1_sorted = fa_loadings['F1'].sort_values(ascending=True)
ax.barh(range(len(f1_sorted)), f1_sorted,
        color=['#4CC9F0' if v>0 else '#F72585' for v in f1_sorted])
ax.set_yticks(range(len(f1_sorted))); ax.set_yticklabels(f1_sorted.index, color='#CCCCCC')
ax.axvline(0, color='white', lw=0.8)
ax.axvline(0.5, color='#FFD60A', lw=0.8, ls='--', alpha=0.7, label='|0.5| threshold')
ax.axvline(-0.5, color='#FFD60A', lw=0.8, ls='--', alpha=0.7)
ax.set_title('F1 (37.2%) — Scoring & Paint Presence\n[FA — Varimax Rotation]',
             color='white', fontsize=10)
ax.set_xlabel('Loading', color='#AAAAAA')
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=8)
ax.grid(axis='x', color='#333344', lw=0.4)

plt.suptitle('PC1 vs F1: How PCA and FA Differ in Interpreting the First Dimension',
             color='white', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print("Notice: PC1 loads on virtually EVERY variable (overall volume).")
print("F1 is more specific: 2P/2PA/FG/FGA/PTS dominate; 3P% and ORB are much lower.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#1A1D27')
ax.tick_params(colors='#CCCCCC', labelsize=9)
for sp in ax.spines.values(): sp.set_edgecolor('#333344')

comm_sorted = communalities.sort_values()
bar_colors = ['#F72585' if v < 0.40 else '#4CC9F0' for v in comm_sorted]
ax.barh(range(len(comm_sorted)), comm_sorted, color=bar_colors, edgecolor='#0F1117')
ax.axvline(0.40, color='#FFD60A', ls='--', lw=1.2, label='0.40 threshold')
ax.set_yticks(range(len(comm_sorted))); ax.set_yticklabels(comm_sorted.index, color='#CCCCCC')
ax.set_xlabel('Communality (proportion of variance explained by 3 factors)', color='#AAAAAA')
ax.set_title('Communalities — FA (3 Factors, Varimax)', color='white', fontsize=11)
ax.legend(facecolor='#1A1D27', labelcolor='white', fontsize=9)
ax.grid(axis='x', color='#333344', lw=0.4)
plt.tight_layout()
plt.show()

low_comm = communalities[communalities < 0.40]
print('Variables with low communality (largely unique variance):')
print(low_comm.sort_values().round(3).to_string())
print('\n→ FG%, 3P%, FT% are poorly explained by the 3 factors.')
print('  Shooting efficiency is player-specific (unique), not a shared latent trait.')